# Task 4 — Improvement Cycle

Starting from the baseline scores in Task 3, we run experiments to improve description quality. Each experiment documents what changed, why, and the resulting scores.

In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import (
    DATASET_PATH,
    GENERATION_MODEL,
    NEBIUS_BASE_URL,
    OUTPUT_EXCEL,
)
from shared.constants import CRITERIA, SYSTEM_PROMPT, final_score
from shared.utils import calculate_cost

client = OpenAI(
    base_url=NEBIUS_BASE_URL,
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

df_products = pd.read_excel(DATASET_PATH)
df_baseline = pd.read_excel(OUTPUT_EXCEL)
print(f"{len(df_products)} products, baseline has {len(df_baseline)} rows")

In [ ]:
def build_user_message(row: pd.Series) -> str:
    return (
        f"Product: {row['product_name']}\n"
        f"Attributes: {row['Product_attribute_list']}\n"
        f"Material: {row['material']}\n"
        f"Warranty: {row['warranty']}"
    )


def run_experiment(prompt: str, model: str = GENERATION_MODEL, **gen_kwargs) -> pd.DataFrame:
    """Generate descriptions for all products and return a results DataFrame."""
    defaults = {"temperature": 0.7, "max_tokens": 200}
    defaults.update(gen_kwargs)

    results = []
    for idx, row in df_products.iterrows():
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": build_user_message(row)},
        ]
        start = time.perf_counter()
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            **defaults,
        )
        elapsed_ms = (time.perf_counter() - start) * 1000
        usage = response.usage
        desc = response.choices[0].message.content.strip()
        results.append({
            "product_name": row["product_name"],
            "generated_description": desc,
            "word_count": len(desc.split()),
            "latency_ms": round(elapsed_ms, 1),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens,
        })
        print(f"  [{idx + 1}/{len(df_products)}] {row['product_name']}: {len(desc.split())} words, {elapsed_ms:.0f}ms")

    return pd.DataFrame(results)


def summarize(results_df: pd.DataFrame):
    """Print quick stats for an experiment run."""
    wc = results_df["word_count"]
    in_range = wc.between(50, 90).sum()
    print(f"Word count: mean={wc.mean():.1f}, min={wc.min()}, max={wc.max()}")
    print(f"In 50-90 range: {in_range}/{len(results_df)} ({in_range/len(results_df):.0%})")
    print(f"Latency: mean={results_df['latency_ms'].mean():.0f}ms")

---
## Experiment 1: Improved Prompt with Few-Shot Example

**What changed:** Added a concrete example of a good description and tightened the word-count instruction.

**Why:** The baseline prompt may be too vague for a 9B model. A few-shot example gives a concrete target for length and tone.

In [ ]:
PROMPT_V2 = """You are an e-commerce copywriter. Write a persuasive product description.

Rules:
- Write between 50 and 90 words. This is strict — count carefully before responding.
- Use ONLY the provided product information. Never invent features or specs.
- Tone: friendly, confident, professional. Sound like a real product listing.
- Highlight the most compelling features and materials.
- Mention the warranty if relevant.
- Output only plain text, no headings or bullet points.

Example:
Product: Sony WH-1000XM5 Headphones
Attributes: features: 30-hour battery, adaptive noise cancelling, multipoint connection; weight: 250g
Material: soft-fit leather, lightweight plastic
Warranty: 1-year limited warranty

Description: Experience studio-quality sound wherever you go with the Sony WH-1000XM5. Industry-leading adaptive noise cancelling blocks out distractions, while the soft-fit leather cushions keep you comfortable through 30 hours of battery life. Seamlessly switch between devices with multipoint connection. The lightweight 250g design means you'll barely notice you're wearing them. Backed by a one-year limited warranty, these headphones deliver premium audio without compromise."""

print("Running experiment 1...")
exp1_results = run_experiment(PROMPT_V2)
print()
summarize(exp1_results)

---
## Experiment 2: Lower Temperature

**What changed:** Reduced temperature from 0.7 to 0.3.

**Why:** Lower temperature reduces randomness, which should improve consistency in following the length constraint and reduce hallucination (better Grounding).

In [ ]:
print("Running experiment 2...")
exp2_results = run_experiment(PROMPT_V2, temperature=0.3)
print()
summarize(exp2_results)

---
## Experiment 3: Different Model

**What changed:** Switched to Meta-Llama-3.1-8B-Instruct (the other available model).

**Why:** Different architectures have different strengths. Llama may follow length constraints better or produce more natural phrasing.

In [ ]:
from shared.config import JUDGE_MODEL  # Llama — the other available model

print(f"Running experiment 3 with {JUDGE_MODEL}...")
exp3_results = run_experiment(PROMPT_V2, model=JUDGE_MODEL, temperature=0.3)
print()
summarize(exp3_results)

---
## Compare Experiments

Pick the best run based on the proportion of descriptions in the 50–90 word range and overall quality.

In [ ]:
comparison = pd.DataFrame({
    "experiment": ["Baseline", "Exp1: Few-shot prompt", "Exp2: Low temp", "Exp3: Llama model"],
    "in_range_pct": [
        df_baseline["generated_description"].apply(lambda x: 50 <= len(str(x).split()) <= 90).mean(),
        exp1_results["word_count"].between(50, 90).mean(),
        exp2_results["word_count"].between(50, 90).mean(),
        exp3_results["word_count"].between(50, 90).mean(),
    ],
    "mean_latency_ms": [
        df_baseline["latency_ms"].mean(),
        exp1_results["latency_ms"].mean(),
        exp2_results["latency_ms"].mean(),
        exp3_results["latency_ms"].mean(),
    ],
})
comparison["in_range_pct"] = (comparison["in_range_pct"] * 100).round(1)
comparison["mean_latency_ms"] = comparison["mean_latency_ms"].round(0)
comparison

## Manual Re-evaluation of Best Experiment

Re-rate a sample of descriptions from the best experiment using the Task 1 rubric to confirm improvement.

In [ ]:
# Choose the best experiment result (change variable as needed)
best_results = exp2_results  # <-- update after reviewing comparison table

# Print 5 samples to evaluate
import numpy as np
sample_idx = np.linspace(0, len(best_results) - 1, 5, dtype=int)
for i in sample_idx:
    row = best_results.iloc[i]
    print(f"--- [{i}] {row['product_name']} ({row['word_count']} words) ---")
    print(row["generated_description"])
    print()

## Summary

_Fill in after running experiments:_

| Experiment | What changed | Result |
|------------|-------------|--------|
| Exp 1 | Few-shot prompt | (e.g., improved length compliance from X% to Y%) |
| Exp 2 | Lower temperature (0.3) | (e.g., more consistent, fewer hallucinations) |
| Exp 3 | Different model (Llama) | (e.g., better/worse on fluency, similar on grounding) |

**Best configuration:** (which combo works best and why)

**What helped most:** (prompt? temperature? model?)

**What didn't help:** (anything surprising?)